# database check

In [1]:
from pathlib import Path
import sqlite3
import pandas as pd

PROJECT_ROOT = Path.cwd()
DB_PATH = PROJECT_ROOT / "data" / "institutional_holding.db"

connection = sqlite3.connect(DB_PATH)
print(f"数据库位置：{DB_PATH}")
print(f"数据库存在：{DB_PATH.exists()}")

数据库位置：d:\KimiData\kimi\workspace\institutional_holding_tracker\data\institutional_holding.db
数据库存在：True


## 查看数据库表

In [12]:
tables = pd.read_sql_query(
    """
    SELECT name
    FROM sqlite_master
    WHERE type = 'table'
    ORDER BY name
    """,
    connection,
)

tables

,name
0,alerts
1,daily_prices
2,fund_holdings
3,holder_mappings
4,holding_changes_summary
5,index_components
6,index_holding_summary
7,indices
8,northbound_holdings
9,sqlite_sequence


## 查看指数基本信息

In [17]:
indices = pd.read_sql_query(
    "SELECT * FROM indices ORDER BY index_code",
    connection,
)

indices

,id,index_name,index_code,exchange,component_count,updated_at
0,13,沪深300,000300,sh,300,2026-08-19 10:34:31
1,16,科创50,000688,sh,50,2026-08-19 10:34:31
2,14,中证500,000905,sh,500,2026-08-19 10:34:31
3,15,创业板指,399006,sz,100,2026-08-19 10:34:31


## 查看指数成分股

In [5]:
components = pd.read_sql_query(
    """
    SELECT index_code, stock_code, stock_name, weight, effective_date
    FROM index_components
    ORDER BY index_code, stock_code
    LIMIT 20
    """,
    connection,
)

components

,index_code,stock_code,stock_name,weight,effective_date
0,000300,000001,平安银行,0.433,2026-08-18
1,000300,000002,万科A,0.087,2026-08-18
2,000300,000063,中兴通讯,0.418,2026-08-18
3,000300,000100,TCL科技,0.378,2026-08-18
4,000300,000157,中联重科,0.145,2026-08-18
5,000300,000166,申万宏源,0.162,2026-08-18
6,000300,000301,东方盛虹,0.119,2026-08-18
7,000300,000333,美的集团,1.634,2026-08-18
8,000300,000338,潍柴动力,0.577,2026-08-18
9,000300,000408,藏格矿业,0.243,2026-08-18


In [14]:
component_counts = pd.read_sql_query(
    """
    SELECT index_code, COUNT(DISTINCT stock_code) AS stock_count
    FROM index_components
    GROUP BY index_code
    ORDER BY index_code
    """,
    connection,
)

component_counts

,index_code,stock_count
0,000300,300


## 统计所有指数并随机查看成分股

In [18]:
all_components = pd.read_sql_query(
    """
    SELECT index_code, stock_code, stock_name, weight, effective_date
    FROM index_components
    ORDER BY index_code, stock_code
    """,
    connection,
)

index_summary = (
    all_components.groupby("index_code", as_index=False)
    .agg(stock_count=("stock_code", "nunique"))
    .sort_values("index_code")
)

print(f"实际写入的指数数量：{len(index_summary)}")
index_summary

实际写入的指数数量：4


,index_code,stock_count
0,000300,300
1,000688,50
2,000905,500
3,399006,100


In [7]:
random_samples = pd.concat(
    [
        group.sample(n=min(5, len(group)), random_state=42)
        for _, group in all_components.groupby("index_code")
    ],
    ignore_index=True,
).sort_values(["index_code", "stock_code"])

random_samples

,index_code,stock_code,stock_name,weight,effective_date
3,000300,000408,藏格矿业,0.243,2026-08-18
2,000300,600372,中航机载,0.105,2026-08-18
0,000300,601077,渝农商行,0.138,2026-08-18
4,000300,601633,长城汽车,0.080,2026-08-18
1,000300,603019,中科曙光,0.469,2026-08-18


## 诊断指数基本信息写入

In [8]:
from config.settings import TRACKED_INDICES
from database.db_manager import query_sql
from ingestion.index_components import _update_indices_info

print("配置中的指数：")
for name, info in TRACKED_INDICES.items():
    print(name, info)

_update_indices_info()

updated_indices = pd.DataFrame(query_sql(
    "SELECT * FROM indices ORDER BY index_code"
))
updated_indices

配置中的指数：
沪深300 {'code': '000300', 'exchange': 'sh'}
中证500 {'code': '000905', 'exchange': 'sh'}
创业板指 {'code': '399006', 'exchange': 'sz'}
科创50 {'code': '000688', 'exchange': 'sh'}


,id,index_name,index_code,exchange,component_count,updated_at
0,1,沪深300,000300,sh,300,2026-08-19 09:29:39
1,4,科创50,000688,sh,0,2026-08-19 09:29:39
2,2,中证500,000905,sh,0,2026-08-19 09:29:39
3,3,创业板指,399006,sz,0,2026-08-19 09:29:39


## 回滚本次诊断写入

In [10]:
index_codes = tuple(info["code"] for info in TRACKED_INDICES.values())
placeholders = ", ".join("?" for _ in index_codes)

connection.execute(
    f"DELETE FROM indices WHERE index_code IN ({placeholders})",
    index_codes,
)
connection.commit()

remaining_indices = pd.read_sql_query(
    "SELECT * FROM indices ORDER BY index_code",
    connection,
)
remaining_components = pd.read_sql_query(
    """
    SELECT index_code, COUNT(DISTINCT stock_code) AS stock_count
    FROM index_components
    GROUP BY index_code
    ORDER BY index_code
    """,
    connection,
)

print(f"回滚后 indices 记录数：{len(remaining_indices)}")
print("成分股明细统计：")
remaining_components

回滚后 indices 记录数：0
成分股明细统计：


,index_code,stock_count
0,000300,300


# 小样本采集测试

## 数据表

In [22]:
table_check = pd.read_sql_query(
    """
    SELECT name
    FROM sqlite_master
    WHERE type = 'table' AND name = 'top_holders'
    """,
    connection,
)

table_check

,name
0,top_holders


## 单只股票测试接口

In [19]:
from ingestion.top_holders import fetch_top_holders_em

df = fetch_top_holders_em("000001", "20250331", tRUE)

print("shape:", df.shape)
print("columns:", list(df.columns))
display(df.head(10))

shape: (10, 7)
columns: ['名次', '股东名称', '股份类型', '持股数', '占总股本持股比例', '增减', '变动比率']


,名次,股东名称,股份类型,持股数,占总股本持股比例,增减,变动比率
0,1,中国平安保险(集团)股份有限公司-集团本级-自有资金,流通A股,9618540236,49.56,不变,NaN
1,2,中国平安人寿保险股份有限公司-自有资金,流通A股,1186100488,6.11,不变,NaN
2,3,香港中央结算有限公司,流通A股,658114653,3.39,-88767070,-11.885024
3,4,中国平安人寿保险股份有限公司-传统-普通保险产品,流通A股,440478714,2.27,不变,NaN
4,5,中国证券金融股份有限公司,流通A股,429232688,2.21,不变,NaN
5,6,中国工商银行股份有限公司-华泰柏瑞沪深300交易型开放式指数证券投资基金,流通A股,158803503,0.82,-8714000,-5.201844
6,7,中国建设银行股份有限公司-易方达沪深300交易型开放式指数发起式证券投资基金,流通A股,110935144,0.57,-4615700,-3.994519
7,8,中国工商银行股份有限公司-华夏沪深300交易型开放式指数证券投资基金,流通A股,75280777,0.39,-1530300,-1.992291
8,9,中国银行股份有限公司-嘉实沪深300交易型开放式指数证券投资基金,流通A股,70005962,0.36,-2766300,-3.801311
9,10,深圳中电投资有限公司,流通A股,62523366,0.32,不变,NaN


In [21]:
from ingestion.top_holders import ingest_all_top_holders

ingest_all_top_holders(
    ["000001"],
    ["20250331", "20241231"],
)

2026-08-19 19:52:41 [INFO] ingestion.top_holders: [TopHolders] Start ingesting 十大股东 for 1 stocks x 2 dates...
2026-08-19 19:52:43 [INFO] ingestion.top_holders: [TopHolders] Ingestion completed. Total records: 20
2026-08-19 19:52:43 [INFO] ingestion.top_holders: [TopHolders] Start ingesting 十大流通股东 for 1 stocks x 2 dates...
2026-08-19 19:52:44 [INFO] ingestion.top_holders: [TopHolders] Ingestion completed. Total records: 20


In [ ]:
# 获取沪深300（000300）300只成分股的十大股东数据
top_holders_schema = pd.read_sql_query(
    "PRAGMA table_info(top_holders)",
    connection,
)

display(top_holders_schema)

top_holders_50 = pd.read_sql_query(
    """
    SELECT th.*
    FROM top_holders AS th
    INNER JOIN (
        SELECT DISTINCT stock_code
        FROM index_components
        WHERE index_code = '000300'
    ) AS c
        ON c.stock_code = th.stock_code
    ORDER BY th.stock_code
    """,
    connection,
)

print(f"获取记录数：{len(top_holders_50)}")
print(f"覆盖股票数：{top_holders_50['stock_code'].nunique()}")

display(top_holders_50)

,cid,name,type,notnull,dflt_value,pk
0,0,id,INTEGER,0,NaN,1
1,1,stock_code,TEXT,1,NaN,0
2,2,stock_name,TEXT,0,NaN,0
3,3,report_date,DATE,1,NaN,0
4,4,holder_name,TEXT,1,NaN,0
5,5,holder_type,TEXT,0,NaN,0
6,6,holder_type_raw,TEXT,0,NaN,0
7,7,hold_shares,REAL,0,NaN,0
8,8,hold_ratio_total,REAL,0,NaN,0
9,9,hold_ratio_float,REAL,0,NaN,0


获取记录数：0
覆盖股票数：0


,id,stock_code,stock_name,report_date,holder_name,holder_type,holder_type_raw,hold_shares,hold_ratio_total,hold_ratio_float,change_status,change_shares,change_ratio,rank,is_float_holder,announce_date,data_source,created_at
